In [31]:
import json
import numpy as np
np.set_printoptions(suppress=True, precision=4, formatter=None)

import numpy as np
from scipy.spatial.transform import Rotation as R

def convert_to_4x4(cam_R_m2c, cam_t_m2c, scale=1):
    # Convert rotation list to a 3x3 matrix
    R = np.array(cam_R_m2c).reshape(3, 3)

    # Scale translation vector
    t = np.array(cam_t_m2c) * scale

    # Create a 4x4 matrix
    T = np.eye(4)
    T[:3, :3] = R  # Rotation part
    T[:3, 3] = t   # Translation part

    return T

def compute_matrix(data):
    """
    Computes the 4x4 matrix from the given JSON data.

    Parameters:
        data (dict): A dictionary containing 'cam_R_m2c', 'cam_t_m2c', and 'obj_id'.

    Returns:
        tuple: A tuple containing the 4x4 matrix and the object ID.
    """
    # Extract rotation and translation
    cam_R_m2c = data.get('cam_R_m2c')
    cam_t_m2c = data.get('cam_t_m2c')
    obj_id = data.get('obj_id')

    if cam_R_m2c is None or cam_t_m2c is None or obj_id is None:
        raise ValueError("JSON data must contain 'cam_R_m2c', 'cam_t_m2c', and 'obj_id'.")

    # Reshape rotation matrix
    R = np.array(cam_R_m2c).reshape(3, 3)

    # Translation vector
    t = np.array(cam_t_m2c).reshape(3, 1)

    # Construct the 4x4 transformation matrix
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()

    return T, obj_id

def process_scene_gt(json_data):
    # Load JSON data
    data = json.loads(json_data)

    # Compute transformation matrix and get obj_id
    T, obj_id = compute_matrix(data)

    # Print with comma-separated values
    formatted_matrix = np.array2string(T, separator=', ')

    print(f"Object ID {obj_id}:")
    print(formatted_matrix)

    return T

def rotation_angle_deg(R1, R2):
    """Geodesic angle between two rotation matrices (degrees)."""
    R_delta = R2 @ R1.T
    # Clamp for numerical stability
    c = np.clip((np.trace(R_delta) - 1.0) / 2.0, -1.0, 1.0)
    return float(np.degrees(np.arccos(c)))

def pose_errors(T1, T2):
    R1, R2 = T1[:3, :3], T2[:3, :3]
    ang_deg1 = rotation_angle_deg(R1, R2)
    print(ang_deg1)

    t1, t2 = T1[:3, 3],  T2[:3, 3]
    R1 = R.from_matrix(T1[:3, :3])
    R2 = R.from_matrix(T2[:3, :3])

    R_delta = R2 * R1.inv()
    ang_deg = np.degrees(R_delta.magnitude())

    eucli_dist = float(np.linalg.norm(t2 - t1))
    return ang_deg, eucli_dist

In [34]:
def image_000008_obj_000001():
    json_data = '''
    {"cam_R_m2c": [0.89556014, 0.43540977, 0.09319581, 0.43772791, -0.82255369, -0.36341162, -0.08157829, 0.36621224, -0.92709042], "cam_t_m2c": [-24.87779034, -15.64017889, 1097.18626032], "obj_id": 1}
    '''

    gt_pose = process_scene_gt(json_data)
    annotated_pose = np.array([[0.9114, 0.4089, 0.0468, -24.7800],
                              [0.4027, -0.8626, -0.3063, -17.2300],
                              [-0.0849, 0.2980, -0.9508, 1106.5400],
                              [0.0000, 0.0000, 0.0000, 1.0000]])

    ang_deg, eucli_dist = pose_errors(annotated_pose, gt_pose)
    print(f"Angular distance (deg): {ang_deg:.2f}, Euclidean distance (mm): {eucli_dist:.2f}")


In [46]:
def image_000008_obj_000010():
  json_data = '''
  {"cam_R_m2c": [0.01937527, 0.98543226, 0.16896244, 0.95816396, 0.02996707, -0.28464996, -0.28556668, 0.16740704, -0.94362351], "cam_t_m2c": [189.88144071, -62.47030265, 1160.3043822], "obj_id": 10}
  '''

  gt_pose = process_scene_gt(json_data)
  annotated_pose = np.array([[0.0553, 0.9935, 0.0994, 187.1882],
                            [0.9614, -0.0261, -0.2740, -62.5707],
                            [-0.2697, 0.1107, -0.9566, 1146.5400],
                            [0.0000, 0.0000, 0.0000, 1.0000]])

  ang_deg, eucli_dist = pose_errors(annotated_pose, gt_pose)
  print(f"Angular distance (deg): {ang_deg:.2f}, Euclidean distance (mm): {eucli_dist:.2f}")

In [42]:
def image_000008_obj_000009():
  json_data = '''
  {"cam_R_m2c": [-0.97578918, -0.19233277, 0.104314, -0.21413325, 0.93739175, -0.27472958, -0.0449417, -0.29041038, -0.95586327], "cam_t_m2c": [-106.51005188, 250.24837343, 1002.06058522], "obj_id": 9}
  '''

  gt_pose = process_scene_gt(json_data)
  annotated_pose = np.array([[-0.9922, -0.0902, 0.0862, -105.6925],
                            [-0.1143, 0.9342, -0.3378, 252.1267],
                            [-0.0501, -0.3450, -0.9373, 1006.5400],
                            [0.0000, 0.0000, 0.0000, 1.0000]])

  ang_deg, eucli_dist = pose_errors(annotated_pose, gt_pose)
  print(f"Angular distance (deg): {ang_deg:.2f}, Euclidean distance (mm): {eucli_dist:.2f}")

In [38]:
def image_000008_obj_000008():
  json_data = '''
  {"cam_R_m2c": [0.46235101, 0.87980556, 0.11901059, 0.86143803, -0.41206849, -0.29999073, -0.21468458, 0.24090753, -0.94751563], "cam_t_m2c": [-71.92675656, 22.59198434, 1015.10816369], "obj_id": 8}
  '''

  gt_pose = process_scene_gt(json_data)
  annotated_pose = np.array([[0.4806, 0.8723, 0.0897, -69.4300],
                            [0.8433, -0.4317, -0.3201, 23.8133],
                            [-0.2406, 0.2295, -0.9431, 1026.5400],
                            [0.0000, 0.0000, 0.0000, 1.0000]])

  ang_deg, eucli_dist = pose_errors(annotated_pose, gt_pose)
  print(f"Angular distance (deg): {ang_deg:.2f}, Euclidean distance (mm): {eucli_dist:.2f}")


In [36]:
def image_000008_obj_000005():
    json_data = '''
    {"cam_R_m2c": [0.95727271, 0.27394454, 0.09303266, 0.28923541, -0.91350467, -0.28619722, 0.00658449, 0.30086648, -0.95367206], "cam_t_m2c": [-35.06928638, 168.30972002, 978.15874479], "obj_id": 5}
    '''

    gt_pose = process_scene_gt(json_data)
    annotated_pose = np.array([[0.9671, 0.2441, 0.0722, -37.4400],
                              [0.2544, -0.9157, -0.3112, 166.7612],
                              [-0.0098, 0.3193, -0.9476, 986.5400],
                              [0.0000, 0.0000, 0.0000, 1.0000]])

    ang_deg, eucli_dist = pose_errors(annotated_pose, gt_pose)
    print(f"Angular distance (deg): {ang_deg:.2f}, Euclidean distance (mm): {eucli_dist:.2f}")

In [24]:
def image_000008_obj_000006():
    gt_pose = process_scene_gt()
    annotated_pose = np.array([[0.2716, -0.9614, -0.0433, -81.7160],
                              [-0.9546, -0.2634, -0.1393, -233.5716],
                              [0.1225, 0.0792, -0.9893, 1156.5388],
                              [0.0000, 0.0000, 0.0000, 1.0000]])

    ang_deg, eucli_dist = pose_errors(annotated_pose, gt_pose)
    print(f"Angular distance (deg): {ang_deg:.2f}, Euclidean distance (mm): {eucli_dist:.2f}")


In [47]:
if __name__ == "__main__":
  # image_000008_obj_000006()
  # image_000008_obj_000001()
  # image_000008_obj_000005()
  # image_000008_obj_000008()
  # image_000008_obj_000009()
  image_000008_obj_000010()


Object ID 10:
[[   0.0194,    0.9854,    0.169 ,  189.8814],
 [   0.9582,    0.03  ,   -0.2846,  -62.4703],
 [  -0.2856,    0.1674,   -0.9436, 1160.3044],
 [   0.    ,    0.    ,    0.    ,    1.    ]]
4.6193080058609715
Angular distance (deg): 4.64, Euclidean distance (mm): 14.03
